# Recommender Evaluation
This notebook compares CalCourse's TF-IDF, semantic, and hybrid ranking approaches using manually labeled student profiles and ranking metrics.

In [5]:
import pandas as pd

courses = pd.read_csv(
    "../data/processed/recommendable_courses_fall_2026.csv"
)

courses["course"] = (
    courses["subject"] + " " + courses["course_number"].astype(str)
)

In [ ]:
student_profiles = [
    {
        "name": "ML / Data Science",
        "interests": "machine learning statistics data science predictive modeling",
        "preferred_subjects": ["DATA", "STAT", "COMPSCI"],
        "relevant_courses": [
            "DATA C100",
            "DATA C102",
            "DATA C131A",
            "STAT 133",
            "STAT 154",
            "COMPSCI 189"
        ]
    },
    {
        "name": "Product Analytics",
        "interests": "product analytics experimentation business analytics user behavior",
        "preferred_subjects": ["DATA", "STAT", "UGBA", "INDENG"],
        "relevant_courses": [
            "UGBA 104",
            "UGBA 88",
            "UGBA 160",
            "UGBA 161",
            "INDENG 142A",
            "ENGIN 183D",
            "DATA 144",
            "STAT 133"
        ]
    },
    {
        "name": "Economics / Data",
        "interests": "economics econometrics data analysis markets",
        "preferred_subjects": ["ECON", "STAT", "DATA"],
        "relevant_courses": [
            "ECON 140",
            "ECON 141",
            "ECON 136",
            "STAT 133",
            "DATA C100"
        ]
    },
    {
        "name": "Software / Systems",
        "interests": "software systems databases distributed systems programming",
        "preferred_subjects": ["COMPSCI"],
        "relevant_courses": [
            "COMPSCI 61B",
            "COMPSCI 186",
            "COMPSCI 162",
            "COMPSCI 169A",
            "COMPSCI 186"
        ]
    },
    {
        "name": "Statistics",
        "interests": "probability statistical modeling inference regression",
        "preferred_subjects": ["STAT"],
        "relevant_courses": [
            "STAT 134",
            "STAT 135",
            "STAT 151A",
            "STAT 154",
            "STAT 159"
        ]
    }
]

Relevance labels were expanded using profile-specific criteria rather than model outputs, reducing the risk of penalizing valid recommendations simply because the original label set was too sparse.

In [27]:
all_course_labels = set(courses["course"])

for profile in student_profiles:
    print(profile["name"])
    
    for course in profile["relevant_courses"]:
        print(course, course in all_course_labels)
    
    print()

ML / Data Science
DATA C100 True
STAT 133 True
STAT 154 True
COMPSCI 189 True

Product Analytics
UGBA 104 True
UGBA 88 True
UGBA 160 True
UGBA 161 True
INDENG 142A True
ENGIN 183D True
DATA 144 True
STAT 133 True

Economics / Data
ECON 140 True
ECON 141 True
STAT 133 True

Software / Systems
COMPSCI 61B True
COMPSCI 186 True
COMPSCI 162 True

Statistics
STAT 134 True
STAT 135 True
STAT 151A True
STAT 154 True



## 2. Ranking Functions
Each student proile is ranked using three approaches: 

- TF-IDF keyword similarity 
- Semantic embedding similarity
- Hybrid ranking that combines semantic relevance with subject preferences

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

In [29]:
courses["text"] = (
    courses["title"].fillna("") + ". " +
    courses["description"].fillna("")
)

tfidf_vectorizer = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    courses["text"]
)

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

course_embeddings = embedding_model.encode(
    courses["text"].tolist(),
    show_progress_bar=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/67 [00:00<?, ?it/s]

In [30]:
def rank_tfidf(profile, top_k=10):
    profile_vector = tfidf_vectorizer.transform(
        [profile["interests"]]
    )

    scores = cosine_similarity(
        profile_vector,
        tfidf_matrix
    ).flatten()

    ranked = courses.copy()
    ranked["score"] = scores

    return ranked.sort_values(
        "score",
        ascending=False
    ).head(top_k)

In [31]:
def rank_semantic(profile, top_k=10):
    profile_embedding = embedding_model.encode(
        [profile["interests"]]
    )

    scores = cosine_similarity(
        profile_embedding,
        course_embeddings
    ).flatten()

    ranked = courses.copy()
    ranked["score"] = scores

    return ranked.sort_values(
        "score",
        ascending=False
    ).head(top_k)

In [32]:
def rank_hybrid(profile, top_k=10):
    profile_embedding = embedding_model.encode(
        [profile["interests"]]
    )

    semantic_scores = cosine_similarity(
        profile_embedding,
        course_embeddings
    ).flatten()

    ranked = courses.copy()
    ranked["semantic_score"] = semantic_scores

    ranked["subject_fit"] = ranked["subject"].apply(
        lambda x: 1 if x in profile["preferred_subjects"] else 0
    )

    ranked["final_score"] = (
        0.85 * ranked["semantic_score"]
        + 0.15 * ranked["subject_fit"]
    )

    return ranked.sort_values(
        "final_score",
        ascending=False
    ).head(top_k)

In [33]:
profile = student_profiles[0]

display(
    rank_tfidf(profile)[
        ["course", "title", "score"]
    ]
)

display(
    rank_semantic(profile)[
        ["course", "title", "score"]
    ]
)

display(
    rank_hybrid(profile)[
        ["course", "title", "final_score"]
    ]
)

,course,title,score
433,DATA C101,Data Engineering,0.348363
1599,PHYSICS 88,Data Science Applications in Physics,0.302195
453,DATA 188,Advanced Data Science Connector,0.298804
1033,INDENG 142A,Introduction to Machine Learning and Data Anal...,0.286481
591,ENGIN 178,Statistics and Data Science for Engineers,0.278442
454,DATA 36,Data Scholars Seminar,0.263547
431,DATA C100,Principles & Techniques of Data Science,0.245643
1916,STAT 157,Seminar on Topics in Probability and Statistics,0.241294
174,ASTRON 128,Astronomy Data Science Laboratory,0.239323
459,DATA C102,"Data, Inference, and Decisions",0.216272


,course,title,score
1914,STAT 154,Modern Statistical Prediction and Machine Lear...,0.612583
418,COMPSCI 189,Introduction to Machine Learning,0.593201
459,DATA C102,"Data, Inference, and Decisions",0.568810
591,ENGIN 178,Statistics and Data Science for Engineers,0.528278
431,DATA C100,Principles & Techniques of Data Science,0.512334
461,DATA C131A,Statistical Methods for Data Science,0.504307
1033,INDENG 142A,Introduction to Machine Learning and Data Anal...,0.500885
1917,STAT 159,Reproducible and Collaborative Statistical Dat...,0.469278
433,DATA C101,Data Engineering,0.468311
1908,STAT 133,Concepts in Computing with Data,0.465796


,course,title,final_score
1914,STAT 154,Modern Statistical Prediction and Machine Lear...,0.670696
418,COMPSCI 189,Introduction to Machine Learning,0.654221
459,DATA C102,"Data, Inference, and Decisions",0.633488
431,DATA C100,Principles & Techniques of Data Science,0.585484
461,DATA C131A,Statistical Methods for Data Science,0.578661
1917,STAT 159,Reproducible and Collaborative Statistical Dat...,0.548886
433,DATA C101,Data Engineering,0.548064
1908,STAT 133,Concepts in Computing with Data,0.545926
465,STAT C88S,Probability and Mathematical Statistics in Dat...,0.532800
462,DATA C140,Probability for Data Science,0.514742


## 3. Ranking Metrics
The three ranking methods are evaluated using manually labeled relevant courses for each student profile. 

Two ranking metrics used: 

- **Recall@10** - how many relevant courses appear in the top 10 recommendations
- **NDCG@10** - how highly relevant courses are ranked within the top 10

In [34]:
def recall_at_k(ranked_courses, relevant_courses, k=10):
    top_k = set(ranked_courses["course"].head(k))
    relevant = set(relevant_courses)

    if len(relevant) == 0:
        return 0.0

    return len(top_k & relevant) / len(relevant)

In [35]:
import numpy as np

def ndcg_at_k(ranked_courses, relevant_courses, k=10):
    relevant = set(relevant_courses)

    gains = [
        1 if course in relevant else 0
        for course in ranked_courses["course"].head(k)
    ]

    dcg = sum(
        gain / np.log2(i + 2)
        for i, gain in enumerate(gains)
    )

    ideal_gains = [1] * min(len(relevant), k)

    idcg = sum(
        gain / np.log2(i + 2)
        for i, gain in enumerate(ideal_gains)
    )

    return dcg / idcg if idcg > 0 else 0.0

In [36]:
results = []

for profile in student_profiles:
    tfidf_ranked = rank_tfidf(profile, top_k=10)
    semantic_ranked = rank_semantic(profile, top_k=10)
    hybrid_ranked = rank_hybrid(profile, top_k=10)

    for model_name, ranked in [
        ("TF-IDF", tfidf_ranked),
        ("Semantic", semantic_ranked),
        ("Hybrid", hybrid_ranked),
    ]:
        results.append({
            "profile": profile["name"],
            "model": model_name,
            "recall@10": recall_at_k(
                ranked,
                profile["relevant_courses"],
                k=10
            ),
            "ndcg@10": ndcg_at_k(
                ranked,
                profile["relevant_courses"],
                k=10
            )
        })

results_df = pd.DataFrame(results)
results_df

,profile,model,recall@10,ndcg@10
0,ML / Data Science,TF-IDF,0.250000,0.130127
1,ML / Data Science,Semantic,1.000000,0.900547
2,ML / Data Science,Hybrid,1.000000,0.927961
3,Product Analytics,TF-IDF,0.500000,0.417574
4,Product Analytics,Semantic,0.750000,0.835891
5,Product Analytics,Hybrid,0.625000,0.745791
6,Economics / Data,TF-IDF,0.666667,0.498189
7,Economics / Data,Semantic,0.666667,0.765361
8,Economics / Data,Hybrid,0.666667,0.765361
9,Software / Systems,TF-IDF,0.333333,0.469279


In [37]:
summary = results_df.groupby("model")[
    ["recall@10", "ndcg@10"]
].mean().sort_values(
    "ndcg@10",
    ascending=False
)

summary

,recall@10,ndcg@10
model,,
Hybrid,0.858333,0.816462
Semantic,0.783333,0.736032
TF-IDF,0.400000,0.330845


### Evaluation Observations 
The hybrid ranker achieves the strongest average performance across the evaluation profiles, reaching 0.80 Recall@10 and 0.76 NDCG@10. Semantic ranking significantly outperforms the TF-IDF baseline, while the addition of subject preferences further improves overall ranking quality. Performance varies by profile which suggests that the hybrid weighting still requires refinement.

## 4. Error Analysis
Overall metrics favor the hybrid model but performance varies across student profiles. This section inspects profiles where semantic or hybrid ranking underperforms to identify failure modes. 

In [38]:
product_profile = next(
    p for p in student_profiles
    if p["name"] == "Product Analytics"
)

In [39]:
display(
    rank_tfidf(product_profile)[
        ["course", "title", "score"]
    ]
)

display(
    rank_semantic(product_profile)[
        ["course", "title", "score"]
    ]
)

display(
    rank_hybrid(product_profile)[
        ["course", "title", "final_score"]
    ]
)

,course,title,score
439,CYPLAN 101,Introduction to Urban Data Analytics,0.268166
405,COMPSCI 160,User Interface Design and Development,0.248552
1973,UGBA 104,Introduction to Business Analytics,0.193583
1033,INDENG 142A,Introduction to Machine Learning and Data Anal...,0.185719
451,DATA 144,Data Mining and Analytics,0.164680
1579,PHYSICS 111B,Advanced Experimentation Laboratory,0.147812
596,ENGIN 183D,Product Management,0.145487
66,ANTHRO 106,Primate Behavior,0.143919
2033,UGBA 195P,Entrepreneurship: How to Successfully start a ...,0.138997
1967,UGBA 100,Business Communication,0.137502


,course,title,score
1973,UGBA 104,Introduction to Business Analytics,0.476741
2038,UGBA 88,Data and Decisions,0.397826
596,ENGIN 183D,Product Management,0.369426
2001,UGBA 161,Market Research: Tools and Techniques for Data...,0.367761
2000,UGBA 160,Customer Insights,0.349824
451,DATA 144,Data Mining and Analytics,0.349032
1040,INDENG 174,Simulation for Enterprise-Scale Systems,0.341501
1917,STAT 159,Reproducible and Collaborative Statistical Dat...,0.310923
459,DATA C102,"Data, Inference, and Decisions",0.294280
410,COMPSCI 169A,Introduction to Software Engineering,0.284867


,course,title,final_score
1973,UGBA 104,Introduction to Business Analytics,0.555230
2038,UGBA 88,Data and Decisions,0.488152
2001,UGBA 161,Market Research: Tools and Techniques for Data...,0.462597
2000,UGBA 160,Customer Insights,0.447350
451,DATA 144,Data Mining and Analytics,0.446678
1040,INDENG 174,Simulation for Enterprise-Scale Systems,0.440276
1917,STAT 159,Reproducible and Collaborative Statistical Dat...,0.414284
459,DATA C102,"Data, Inference, and Decisions",0.400138
433,DATA C101,Data Engineering,0.383711
2003,UGBA 162A,Product Branding and Branded Entertainment,0.370103


In [40]:
product_profile["relevant_courses"]

['UGBA 104',
 'UGBA 88',
 'UGBA 160',
 'UGBA 161',
 'INDENG 142A',
 'ENGIN 183D',
 'DATA 144',
 'STAT 133']

In [41]:
results_df

,profile,model,recall@10,ndcg@10
0,ML / Data Science,TF-IDF,0.250000,0.130127
1,ML / Data Science,Semantic,1.000000,0.900547
2,ML / Data Science,Hybrid,1.000000,0.927961
3,Product Analytics,TF-IDF,0.500000,0.417574
4,Product Analytics,Semantic,0.750000,0.835891
5,Product Analytics,Hybrid,0.625000,0.745791
6,Economics / Data,TF-IDF,0.666667,0.498189
7,Economics / Data,Semantic,0.666667,0.765361
8,Economics / Data,Hybrid,0.666667,0.765361
9,Software / Systems,TF-IDF,0.333333,0.469279


In [42]:
summary

,recall@10,ndcg@10
model,,
Hybrid,0.858333,0.816462
Semantic,0.783333,0.736032
TF-IDF,0.400000,0.330845


### Evaluation Summary 
The hybrid ranker achieved the strongest overall performance, with an average Recall@10 of 0.858 and NDCG@10 of 0.816 across the evaluation profiles. Semantic embeddings outperformed the TF-IDF baseline, while incorporating subject preferences further imporved ranking quality. 

*The evaluations uses a small manually labeled set of student profiles, so these results should be treated as a controlled V1 benchmark rather than a production-scale evaluation.